# Demo: Retrieval y Sistema Agéntico

Este notebook demuestra las diferentes estrategias de retrieval y el sistema agéntico.

In [1]:
import sys
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv("../env/.env")

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval.vector_retriever import VectorRetriever, HybridRetriever
from graphrag.retrieval.fulltext_retriever import FullTextRetriever
from graphrag.retrieval.text2cypher import Text2CypherRetriever
from graphrag.agents import AgenticRAG

## 1. Inicializar conexión

In [2]:
neo4j = Neo4jManager()

## 2. Probar Vector Retrieval

In [4]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("What is the average lifespan of a lion?")

print("Vector Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text']}")
    print(f"   {result['matched_questions']})")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4587.41it/s]
Accessing `__iter__` from `.models.aria.image_processing_aria`. Returning `__iter__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__iter__` from `.models.auto.image_processing_auto`. Returning `__iter__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__iter__` from `.models.beit.image_processing_beit`. Returning `__iter__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__iter__` from `.models.bit.image_processing_bit`. Returning `__iter__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__iter__` from `.models.blip.image_processing_blip`. Returning `__iter__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__iter__` from `.models.bridgetower.image_processing_bridgetower`. 

Vector Search Results:

1. Score: 0.975
   Animal: Lion
Section: Reproduction, Cubs, and Lifespan
Sadly, however, less than half of cubs make it to be a year old and four out of five have died by the time they are two, generally either from animal attacks or starvation. Remarkably though, the female lions in the pride will have their cubs at around the same time and will help to suckle and care for the cubs of other females.  
Lion cubs suckle on milk until they are about six months old and although they won’t begin actively hunting until they are about a year old, lion cubs start to eat meat after 12 weeks or so.  
Like most big cats, lions live about 10 to 15 years. In captivity, lions have lived quite a bit longer than in the wild. In 2016, the Philadelphia Zoo had to euthanize a 25-year-old female lion after it began suffering from limited mobility.
   ['What is the typical lifespan of a lion in the wild?', 'Do lions live longer in captivity than in the wild?', 'What percentage of 

## 3. Probar Full-text y Hybrid Retrieval

In [4]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("Albatros location")

print("Full text Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

Full text Search Results:


In [5]:
hybrid_retriever = HybridRetriever(neo4j)

results = hybrid_retriever.retrieve("Einstein Nobel Prize")

print("Hybrid Search Results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.3f}")
    print(f"   {result['text'][:200]}...")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL {\n            // Vector search\n            CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n            YIELD node, score\n            WITH collect({node: node, score: score}) AS nodes, max(score) AS max\n            UNWIND nodes AS n\n            RETURN n.node AS node, (n.score / max) AS score\n   

Hybrid Search Results:

1. Score: 1.000
   Animal: Albatross
Section: Etymology
The name "Albatross" is derived from the Arabic al-qādūs القادوس or al-ḡaṭṭās الغطاس (a pelican; literally, "the diver"), which travelled to English via the Portug...

2. Score: 0.977
   Animal: Albatross
Stakeholders such as governments, conservation organisations, and people in the fishing industry are all working toward reducing this phenomenon....

3. Score: 0.970
   Animal: Albatross
Section: Taxonomy and evolution
Within the family, the assignment of genera has been debated for over 100 years. Originally placed into a single genus, Diomedea, they were rearranged...

4. Score: 0.957
   Animal: Albatross
Albatrosses are highly efficient in the air, using dynamic soaring and slope soaring to cover great distances with little exertion. They feed on squid, fish, and krill by either scav...

5. Score: 0.955
   Animal: Albatross
Section: Taxonomy and evolution
They are Murunkus (Middle Eocene of Uzbekistan),

## 4. Probar Text2Cypher

In [ ]:
text2cypher = Text2CypherRetriever(neo4j)

# Añadir ejemplos few-shot
text2cypher.add_few_shot_example(
    "How many Mammal species are there?",
    "MATCH (s:Species)-[:BELONGS_TO_CLASS]->(c:AnimalClass {type: 'Mammal'}) RETURN count(s) AS totalMammals"
)

cypher, results = text2cypher.retrieve("How many Mammal species are there?")

print("Generated Cypher:")
print(cypher)
print("\nResults:")
for result in results:
    print(result)

## 5. Probar Sistema Agéntico

In [ ]:
agentic_rag = AgenticRAG(neo4j)

# Pregunta simple
result = agentic_rag.answer("Where was Einstein born?")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nIterations:", result['final_critique'])

In [ ]:
# Pregunta compleja que requiere agregación
result = agentic_rag.answer("List all locations mentioned in the documents")

print("Question:", result['question'])
print("\nAnswer:", result['answer'])
print("\nTool used:", result['iterations'][-1]['retrieval']['tool'])
print("\nRouting decision:", result['iterations'][-1]['retrieval']['routing_decision']['reasoning'])

In [ ]:
# Conversación multi-turno
agentic_rag.reset_conversation()

result1 = agentic_rag.answer("What theory did Einstein develop?")
print("Q1:", result1['question'])
print("A1:", result1['answer'])

result2 = agentic_rag.answer("When was it published?")
print("\nQ2:", result2['question'])
print("A2:", result2['answer'])

In [ ]:
neo4j.close()